In [ ]:
# Ô 1 — gói, Java, mốc giờ
import subprocess, sys, os, time
T_NB = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y openjdk-8-jre-headless >/dev/null 2>&1; java -version")
sh(f"{sys.executable} -m pip install -q 'trl==0.29.1' 'transformers==5.18.0' 'peft==0.21.1' accelerate bitsandbytes pycocoevalcap sacrebleu 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y -q torchao")
import importlib.util; assert importlib.util.find_spec("torchao") is None, "DỪNG: torchao còn"
sh(f'{sys.executable} -c "import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0))"')


In [ ]:
# Ô 2 — đường dẫn, md5, hàm chạy song song
import glob, json, shutil, hashlib, re
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
MD5 = {
 "build_branch_data.py": "619e63e123a6dbf60086e65ee94a3912",
 "ex_test_k4.jsonl": "4d8d1646402aeda72f3a826c3d663702",
 "ex_val_k4.jsonl": "3a7ae1e2360659c684234e9d6fbcb866",
 "grpo_spice.py": "07ea87b6d156d1faa391a4274dffd7cf",
 "pred_ck500_test.jsonl": "328847ac96fa3104f76cd997f4091bcd",
 "pred_ck500_vallon.jsonl": "d092faf6f862eec5db71110e6b5963e2",
 "ra_build.py": "7e4ee74274037b83d11db5d63c5e66a8",
 "ra_exemplars.py": "319857d87a6d5fdf8578886ffbb1ac0f",
 "ra_gen.py": "d3969f6105b53b9eae81e7ce521cbb27",
 "ra_score.py": "04aab5e14ee01468b2c6fb8683bc2f8a",
 "ra_train.py": "217c75f188046c4bd279bb11fb1757e4",
 "train_tru_val.jsonl": "edbb231a58574acf6e1968a44d1fa6c6"
}
SRC = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ra_train.py", recursive=True)
       if os.path.exists(os.path.join(os.path.dirname(p), "train_tru_val.jsonl"))]
assert len(SRC) == 1, f"DỪNG: cần đúng một dataset ra-sft-script, thấy {SRC}"
SRC = SRC[0]
for f, h in MD5.items():
    assert md5(f"{SRC}/{f}") == h, f"DỪNG: {f} lệch md5"
    if f.endswith(".py"):
        shutil.copy(f"{SRC}/{f}", f"{W}/{f}")
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "p1_train_rows.jsonl" in f)
CK = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
            if md5(p) == "491fa6677340393f1e4464c08a0cec98")
CK = [c for c in CK if not c.strip("/").endswith("ref")]
assert len(CK) == 1, f"DỪNG: không thấy đúng một adapter ck500, thấy {CK}"
CK500 = CK[0]
MERGED = f"{W}/s1_merged"
def chay(jobs, nhip=120):
    ps = []
    for i, (ten, cmd, gpu) in enumerate(jobs):
        if i: time.sleep(90)
        f = open(f"{W}/{ten}.log", "w")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
        ps.append((ten, subprocess.Popen(cmd, cwd=W, stdout=f, stderr=subprocess.STDOUT, env=env), f))
    while any(p.poll() is None for _, p, _ in ps):
        time.sleep(nhip)
        for ten, p, _ in ps:
            L = [l.strip() for l in open(f"{W}/{ten}.log", errors="ignore") if l.strip()]
            print(f" {(time.time() - T_NB) / 3600:5.2f} h · {ten} · {'xong' if p.poll() is not None else 'chạy'} · {L[-1][:110] if L else '..'}", flush=True)
    for ten, p, f in ps:
        f.close()
        print(f"--- {ten} · mã thoát {p.returncode} ---\n" + "".join(open(f"{W}/{ten}.log", errors="ignore").readlines()[-10:]), flush=True)
    return {ten: p.returncode for ten, p, _ in ps}
def hoa():
    r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", MERGED], cwd=W, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); assert r.returncode == 0, "DỪNG: hoà S1 lỗi"
print("SRC   ", SRC); print("BUNDLE", BUNDLE); print("CK500 ", CK500, flush=True)


In [ ]:
# Ô hoà S1
hoa()
assert os.path.isdir(MERGED)

In [ ]:
# Ô C1-1: chép mã, hàm chạy nền
import os, json, time, shutil, subprocess, glob
W = "/kaggle/working"; RA = f"{W}/ra"; M = f"{W}/s1_merged"

def tim(duoi):
    for sau in range(1, 6):
        g = glob.glob("/kaggle/input/" + "*/" * sau + duoi)
        if g:
            return g[0]
    raise FileNotFoundError(duoi)

CW = os.path.dirname(tim("cw_gen.py"))
shutil.copytree(SRC, RA, dirs_exist_ok=True)
for f in os.listdir(CW):
    shutil.copy(f"{CW}/{f}", RA)
print("CW", CW, sorted(os.listdir(RA)))
T0 = T_NB  # tính giờ từ đầu notebook (gồm pip + hoà S1) để chắc không chạm trần 12 h

def chay(script, args, log, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    return subprocess.Popen(["python", f"{RA}/{script}", *args], stdout=open(log, "w"),
                            stderr=subprocess.STDOUT, env=env, cwd=RA, start_new_session=True)

def doi(ps, logs):
    while any(p.poll() is None for p in ps):
        time.sleep(120)
        print(time.strftime("%H:%M"), f"{(time.time()-T0)/3600:.2f} h",
              [open(l).read().splitlines()[-1:] for l in logs], flush=True)
    assert all(p.returncode == 0 for p in ps), [open(l).read()[-2000:] for l in logs]

In [ ]:
# Ô C2-1: tìm adapter CW, đặt đường dẫn tập kiểm
CWA = os.path.dirname(tim("cw_sft/final/adapter_config.json"))
PKG = "/kaggle/input/datasets/uyenle0712/thesis-score/thesis"
if not os.path.isdir(PKG):
    PKG = os.path.dirname(os.path.dirname(tim("harness/score_run.py")))
TEST = f"{PKG}/harness/dg1_cache/test_ac"
print("CWA", CWA, "\nPKG", PKG)
assert os.path.exists(f"{TEST}/test.jsonl") and os.path.exists(f"{PKG}/harness/score_run.py")

In [ ]:
# Ô sinh chia shard: mỗi GPU chạy tuần tự các việc của nó; nối tiếp được (cw_gen bỏ qua dòng đã có)
VIEC = {0: [('val', 0, 2), ('test', 0, 10), ('test', 1, 10)], 1: [('val', 1, 2), ('test', 2, 10), ('test', 3, 10)]}
def lenh(v):
    ten, k, n = v
    if ten == "val":
        a = [f"{BUNDLE}/p1_val_rows.jsonl", f"{BUNDLE}/ocr.jsonl", f"{BUNDLE}/images", f"{SRC}/ex_val_k4.jsonl", "1002"]
    else:
        a = [f"{TEST}/test.jsonl", f"{TEST}/ocr.jsonl", f"{TEST}/images", f"{SRC}/ex_test_k4.jsonl", "4463"]
    return ["python", f"{RA}/cw_gen.py", "--mode", "cw", "--merged", M, "--ckpt", CWA, "--recs", a[0], "--ocr", a[1],
            "--images", a[2], "--ra", a[3], "--chi-cham", "--expect", a[4],
            "--out", f"{W}/cw_{ten}_s{k}of{n}.jsonl", "--shard", f"{k}/{n}"]
import shlex
ps, logs = [], []
for g in (0, 1):
    cmd = " && ".join(" ".join(map(shlex.quote, lenh(v))) for v in VIEC[g])
    log = f"{W}/sinh_gpu{g}.log"; logs.append(log)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTHONUNBUFFERED="1")
    ps.append(subprocess.Popen(cmd, shell=True, stdout=open(log, "w"), stderr=subprocess.STDOUT,
                               env=env, cwd=RA, start_new_session=True))
    time.sleep(60)
doi(ps, logs)
for g in (0, 1):
    for ten, k, n in VIEC[g]:
        f = f"{W}/cw_{ten}_s{k}of{n}.jsonl"
        print(f, sum(1 for _ in open(f)), flush=True)


In [ ]:
# Ô đóng gói: xoá thứ nặng, zip riêng kết quả
for d in ("s1_merged", "ra"):
    shutil.rmtree(f"{W}/{d}", ignore_errors=True)
import zipfile
with zipfile.ZipFile(f"/tmp/ket_qua_cw-ra-g1-309.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(W)):
        if f.startswith(("cw_", "sinh_", "kiem20")) and f.endswith((".jsonl", ".log")):
            z.write(f"{W}/{f}", f)
shutil.move("/tmp/ket_qua_cw-ra-g1-309.zip", f"{W}/ket_qua_cw-ra-g1-309.zip")
print(sorted(os.listdir(W)), f"{(time.time()-T0)/3600:.2f} h")
